In [53]:
import numpy as np
from scipy.optimize import linprog
import pandas as pd

def optimize_energy_mix_no_storage(D, lambda_=0.5,
                                   import_share_max=0.08,
                                   import_cost=7.5,
                                   import_emissions=0.65):
    # Energy sources (no storage)
    sources = ['Coal', 'Oil', 'Natural Gas', 'Hydro', 'Nuclear',
               'Solar', 'Wind', 'Biomass', 'Energy Imports']
    n_src = len(sources)

    # Base costs (INR/kWh) and emissions (kgCO2/kWh)
    costs = np.array([4.0, 8.5, 6.0, 4.5, 6.5, 2.5, 3.0, 4.5, import_cost])
    emissions = np.array([0.82, 0.78, 0.45, 0.0, 0.0, 0.0, 0.0, 0.05, import_emissions])

    # Apply efficiency factor (0.7 for fossil + biomass)
    eff_factor = np.ones(n_src)
    for i, s in enumerate(sources):
        if s in ['Coal', 'Oil', 'Natural Gas', 'Biomass']:
            eff_factor[i] = 0.7
    costs_eff = costs / eff_factor
    emissions_eff = emissions / eff_factor

    # Hard min/max share constraints (fractions of D)
    min_shares = np.array([
        0.20,  # Coal
        0.20,  # Oil
        0.10,  # Natural Gas
        0.05,  # Hydro
        0.05,  # Nuclear
        0.20,  # Solar
        0.05,  # Wind
        0.05,  # Biomass
        0.00   # Imports
    ])

    max_shares = np.array([
        0.40,  # Coal
        0.30,  # Oil
        0.20,  # Natural Gas
        0.10,  # Hydro
        0.10,  # Nuclear
        0.50,  # Solar
        0.50,  # Wind
        0.10,  # Biomass
        import_share_max
    ])

    # Sanity check
    if min_shares.sum() > 1.0:
        raise ValueError("Sum of minimum shares exceeds 100% — infeasible setup!")

    # Objective function: weighted cost + emissions
    c = lambda_ * costs_eff + (1 - lambda_) * emissions_eff

    # Constraints
    A_eq = np.ones((1, n_src))
    b_eq = [D]

    A_ub, b_ub = [], []

    # Individual min/max bounds
    for i in range(n_src):
        row_max = np.zeros(n_src)
        row_max[i] = 1
        A_ub.append(row_max)
        b_ub.append(max_shares[i] * D)

        row_min = np.zeros(n_src)
        row_min[i] = -1
        A_ub.append(row_min)
        b_ub.append(-min_shares[i] * D)

    # Combined solar + wind ≤ 50% constraint
    solar_idx = sources.index('Solar')
    wind_idx = sources.index('Wind')
    row_sw_upper = np.zeros(n_src)
    row_sw_upper[solar_idx] = 1
    row_sw_upper[wind_idx] = 1
    A_ub.append(row_sw_upper)
    b_ub.append(0.50 * D)

    bounds = [(0, None)] * n_src

    # Solve linear program
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                  bounds=bounds, method='highs')

    if not res.success:
        raise RuntimeError("Optimization failed: " + res.message)

    # Extract results
    x = res.x
    shares = x / D

    results_df = pd.DataFrame({
        'Source': sources,
        'Optimal Supply (GWh)': x / 1e6,
        'Share of Total (%)': shares * 100
    })

    summary = {
        'Total Demand (GWh)': D / 1e6,
        'Total Cost (INR)': np.dot(costs_eff, x),
        'Total Emissions (kg CO2)': np.dot(emissions_eff, x),
        'Lambda': lambda_
    }

    return results_df, summary




In [54]:
def supply_gwh_for_lambdas(D, import_share_max=0.08, import_cost=7.5, import_emissions=0.65):
    lambdas = [0,0.25,0.5,0.75,1]
    supply_data = {}
    for lambda_ in lambdas:
        results_df, _ = optimize_energy_mix(D, lambda_)
        supply_data[f'lambda={lambda_}'] = results_df['Optimal Supply (GWh)'].values

    
    supply_gwh_df = pd.DataFrame(supply_data, index=results_df['Source'])
    return supply_gwh_df



In [55]:
years = [2030, 2035, 2040, 2045, 2047]
demands_gwh = [1888454, 2722928, 4040921, 6119736, 7257287]

table_data = {}

for year, D_gwh in zip(years, demands_gwh):
    D = D_gwh * 1e6  # Convert GWh to kWh (if your model expects kWh)
    supply_df = supply_gwh_for_lambdas(D)  # Make sure lambda=0.5 is used within this function
    table_data[year] = supply_df['lambda=0.5']  # Extract the supply vector for lambda=0.5

final_df = pd.DataFrame(table_data)
final_df.index.name = 'Source'

final_df


RuntimeError: Optimization failed: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is At lower/fixed bound)